# Manual discourse validation - 50 cases

The manual-review workflow for the discopy validation sample. Reads
`manual_validation_sample_50.csv` and writes
`manual_validation_completed.csv`.

**What this notebook does not do.** It does not change parser predictions, the
sampling, or the sample file. It adds no statistical estimator, confidence
interval, recall metric, or automatic A/B/C decision - the evaluation cell at
the end simply calls the existing logic in `evaluate_manual_validation.py`.

**Saving.** Every control writes straight through to
`manual_validation_completed.csv` on change. There is no submit step, and
closing the notebook mid-review loses nothing: re-running the setup cells
reloads whatever was answered.

In [1]:
# ============================================================
# Setup
# ============================================================

from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=None, repo_name="masters_thesis_sdg"):
    current = (start or Path.cwd()).resolve()
    while True:
        if current.name == repo_name:
            return current
        if current.parent == current:
            raise FileNotFoundError(
                f"Could not find repo root {repo_name!r} above {Path.cwd()}"
            )
        current = current.parent


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.sentences import split_sentences
from src.justification_analysis.validation.discourse_review_tools import (
    COMPLETED_COLUMNS, MANUAL_COLUMNS, DiscourseReviewApp, ValidationStore,
    build_cases, progress_summary, render_case_html,
)
from src.justification_analysis.comparison.discourse_statistics import (
    load_justification_frame,
)

ARTIFACTS = (
    REPO_ROOT / "analysis" / "cross_model" / "base" / "voting"
    / "prompt_v4" / "justification_analysis" / "discourse_parser"
)

SAMPLE_PATH = ARTIFACTS / "manual_validation_sample_50.csv"
COMPLETED_PATH = ARTIFACTS / "manual_validation_completed.csv"

print("sample   :", SAMPLE_PATH.name)
print("answers  :", COMPLETED_PATH.name)

sample   : manual_validation_sample_50.csv
answers  : manual_validation_completed.csv


In [2]:
# ============================================================
# Load the sample (read-only) and any answers already given
# ============================================================

sample = pd.read_csv(SAMPLE_PATH, encoding="utf-8-sig")
justifications = load_justification_frame(REPO_ROOT)

cases = build_cases(sample, justifications, split_sentences)
store = ValidationStore(COMPLETED_PATH, cases)

# The sample is fixed. If these ever fail, the sample was regenerated and any
# answers already collected no longer line up with it.
assert len(cases) == 50, f"expected 50 cases, got {len(cases)}"
assert sample["validation_id"].is_unique
assert sample["failure_type"].value_counts().to_dict() == {
    "accepted": 30, "rejected_nosense": 10, "not_enumerated": 10
}, "sample strata changed"

# COMPLETED_COLUMNS is the schema evaluate_manual_validation.py expects.
assert set(MANUAL_COLUMNS) <= set(COMPLETED_COLUMNS)

print(f"cases loaded  : {len(cases)}")
print(f"already answered: {store.n_answered()} / {len(cases)}")
display(sample["failure_type"].value_counts()
        .rename_axis("failure_type").reset_index(name="n"))

cases loaded  : 50
already answered: 50 / 50


,failure_type,n
0,accepted,30
1,rejected_nosense,10
2,not_enumerated,10


## How to review

Run the cell below once; it renders an interactive reviewer.

- **Accepted cases** (blue): is the highlighted span actually functioning as a
  discourse connective, and if so which top-level PDTB category is correct.
- **`rejected_nosense`** (orange) - discopy saw the span and rejected it - and
  **`not_enumerated`** (purple) - discopy never proposed it: is this a valid
  discourse relation discopy failed to report, and if so which category.

The parser's own prediction and confidence are shown. This is targeted
inspection, not blinded annotation.

`Next unanswered` skips ahead; the type-in box plus `Go` jumps to a case
number. Every click saves.

In [3]:
# ============================================================
# Reviewer
# ============================================================

app = DiscourseReviewApp(cases, store)
display(app.ui)

## Progress

Re-run this whenever you want to check where you are. The completed file is
already on disk - this only reports and re-saves it.

In [4]:
store.save()

display(progress_summary(cases, store))

remaining = [c for c in cases if not store.is_answered(c)]
print(f"saved -> {COMPLETED_PATH}")
if remaining:
    print(f"{len(remaining)} case(s) still unanswered: "
          f"{[int(c['validation_id']) for c in remaining][:15]}"
          f"{' ...' if len(remaining) > 15 else ''}")
else:
    print("All 50 cases answered.")

,failure_type,n,answered,remaining
0,accepted,30,30,0
1,rejected_nosense,10,10,0
2,not_enumerated,10,10,0
3,TOTAL,50,50,0


saved -> C:\Users\annab\Documents\GitHub\masters_thesis_sdg\analysis\cross_model\base\voting\prompt_v4\justification_analysis\discourse_parser\manual_validation_completed.csv
All 50 cases answered.


### Static view (optional)

A non-interactive rendering of any single case, for reading or printing. It
does not record anything.

In [5]:
from IPython.display import HTML, display as _display

CASE_TO_SHOW = 1   # 1-50

_display(HTML(render_case_html(
    cases[CASE_TO_SHOW - 1], CASE_TO_SHOW, len(cases)
)))

## Evaluation

Runs the existing logic in `src/justification_analysis/validation/evaluate_manual_validation.py`
unchanged. Raw counts only: no precision estimate, no confidence interval, no
recall, and no automatic A/B/C choice.

**The report is only produced once all 50 cases are answered.** Running it on a
partially completed sheet would produce counts over an arbitrary subset and
invite reading them as results, so the cell refuses and tells you what is left.

In [6]:
# ============================================================
# Validation report
# ============================================================

import subprocess

n_answered = store.n_answered()

if n_answered < len(cases):
    print(f"Not run: {n_answered}/{len(cases)} cases answered.")
    print(f"{len(cases) - n_answered} still to review - finish them above, "
          "then re-run this cell.")
    print("\n(The evaluator is deliberately not run on partial labels.)")
else:
    store.save()
    result = subprocess.run(
        [sys.executable,
         str(REPO_ROOT / "src" / "justification_analysis"
             / "evaluate_manual_validation.py"),
         "--csv", str(COMPLETED_PATH),
         "--out", str(ARTIFACTS)],
        capture_output=True, text=True,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:\n", result.stderr)


STDERR:
 C:\Users\annab\miniconda3\envs\sdglogs\python.exe: can't open file 'C:\\Users\\annab\\Documents\\GitHub\\masters_thesis_sdg\\src\\justification_analysis\\evaluate_manual_validation.py': [Errno 2] No such file or directory



### Where things are

| what | path |
|---|---|
| sample (read-only) | `analysis/.../justification_analysis/discourse_parser/manual_validation_sample_50.csv` |
| your answers | `analysis/.../justification_analysis/discourse_parser/manual_validation_completed.csv` |
| report (once complete) | `analysis/.../justification_analysis/discourse_parser/manual_validation_report.txt` |

To resume after stopping: reopen this notebook and run the setup cells. Answers
are read back from the completed file, and the reviewer picks up where you left
off - `Next unanswered` jumps straight to the first remaining case.